In [2]:
# ============================================================
# E4 — Section-wise Confidence WSFT
# - WSFT token weights × section-specific confidence
# - Sections: decision / explanation / format
# ============================================================

!pip -q install -U transformers peft accelerate bitsandbytes sentencepiece

import os, json, math, re
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model
from google.colab import drive

drive.mount("/content/drive")
# ----------------------------
# CONFIG
# ----------------------------
BASE_DIR = "/content/drive/MyDrive/DL_Local"
PROMPTS_PATH = os.path.join(BASE_DIR, "clinical_pharm_prompts_10000.jsonl")
TEACHER_PATH = os.path.join(BASE_DIR, "clinical_pharm_teacher_train_6000.jsonl")
OUT_DIR = os.path.join(BASE_DIR, "e4_section_cw_wsft")
os.makedirs(OUT_DIR, exist_ok=True)

MAX_SEQ_LEN = 2048
EPOCHS = 2
LR = 2e-4
BATCH = 1
GRAD_ACC = 32
SEED = 42

# WSFT base weights
W_FORMAT = 1.0
W_DEC = 2.0
W_EXPL = 2.0

ALPHA_MIN, ALPHA_MAX = 0.25, 3.0

STUDENTS = {
    "qwen25_1p5b_base":     {"path": "Qwen/Qwen2.5-1.5B",          "is_instruct": False},
    "qwen25_1p5b_instruct": {"path": "Qwen/Qwen2.5-1.5B-Instruct", "is_instruct": True},
    "qwen25_3b_base":       {"path": "Qwen/Qwen2.5-3B",            "is_instruct": False},
    "qwen25_3b_instruct":   {"path": "Qwen/Qwen2.5-3B-Instruct",   "is_instruct": True},
    "qwen25_7b_base":       {"path": "Qwen/Qwen2.5-7B",            "is_instruct": False},
    "qwen25_7b_instruct":   {"path": "Qwen/Qwen2.5-7B-Instruct",   "is_instruct": True},
}


# ----------------------------
# Load data
# ----------------------------
def load_jsonl(p):
    with open(p, "r", encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

prompts = {r["id"]: r["prompt"] for r in load_jsonl(PROMPTS_PATH)}
rows = [r for r in load_jsonl(TEACHER_PATH) if r["status"]=="ok"]

# ----------------------------
# Section parsing
# ----------------------------
DEC_PAT = [r"\bDecision\s*:\s*", r'"decision"\s*:\s*']
EXP_PAT = [r"\bExplanation\s*:\s*", r'"explanation"\s*:\s*']

def find_span(txt, start_pats, end_pats):
    s = None
    for p in start_pats:
        m = re.search(p, txt)
        if m:
            s = m.end()
            break
    if s is None:
        return None
    e = len(txt)
    for p in end_pats:
        m = re.search(p, txt[s:])
        if m:
            e = min(e, s + m.start())
    return (s, e)

def get_spans(ans):
    d = find_span(ans, DEC_PAT, EXP_PAT)
    e = find_span(ans, EXP_PAT, DEC_PAT)
    return d, e

# ----------------------------
# Confidence
# ----------------------------
def sigmoid(x): return 1/(1+math.exp(-x))

def section_conf(row, span):
    steps = row.get("logprobs_steps") or []
    if not span:
        return None
    s,e = span
    margins=[]
    for st in steps:
        ch = st["chosen"]
        top = st["topk"]
        if not top: continue
        alt = max(tc["logprob"] for tc in top if tc["token"]!=ch["token"])
        margins.append(ch["logprob"]-alt)
    if not margins: return None
    return sigmoid(sorted(margins)[len(margins)//2])

# Precompute section means
dec_confs, expl_confs = [], []
for r in rows:
    d,e = get_spans(r["teacher_text"])
    if d:
        c=section_conf(r,d)
        if c: dec_confs.append(c)
    if e:
        c=section_conf(r,e)
        if c: expl_confs.append(c)

MEAN_DEC = sum(dec_confs)/len(dec_confs)
MEAN_EXP = sum(expl_confs)/len(expl_confs)

# ----------------------------
# Dataset
# ----------------------------
class E4Dataset(Dataset):
    def __init__(self, rows, tok, is_instr):
        self.rows, self.tok, self.is_instr = rows, tok, is_instr

    def __len__(self): return len(self.rows)

    def __getitem__(self,i):
        r = self.rows[i]
        prompt = prompts[r["id"]]
        ans = r["teacher_text"]

        if self.is_instr and hasattr(self.tok,"apply_chat_template"):
            ptxt = self.tok.apply_chat_template(
                [{"role":"user","content":prompt}],
                tokenize=False, add_generation_prompt=True
            )
        else:
            ptxt = prompt

        full = ptxt + ans
        enc = self.tok(full, truncation=True, max_length=MAX_SEQ_LEN, return_offsets_mapping=True)
        ids, offs = enc["input_ids"], enc["offset_mapping"]

        labels = [-100]*len(ids)
        weights = [0.0]*len(ids)

        d,e = get_spans(ans)
        if d:
            a_dec = section_conf(r,d)
            a_dec = max(ALPHA_MIN,min(ALPHA_MAX,a_dec/MEAN_DEC)) if a_dec else 1.0
        else: a_dec=1.0
        if e:
            a_exp = section_conf(r,e)
            a_exp = max(ALPHA_MIN,min(ALPHA_MAX,a_exp/MEAN_EXP)) if a_exp else 1.0
        else: a_exp=1.0

        start = len(ptxt)
        for i,(s,_) in enumerate(offs):
            if s>=start:
                labels[i]=ids[i]
                w=W_FORMAT
                if d and start+d[0]<=s<start+d[1]: w=W_DEC*a_dec
                if e and start+e[0]<=s<start+e[1]: w=W_EXPL*a_exp
                weights[i]=w

        return {
            "input_ids":torch.tensor(ids),
            "attention_mask":torch.ones(len(ids)),
            "labels":torch.tensor(labels),
            "weights":torch.tensor(weights),
        }

# ----------------------------
# Trainer
# ----------------------------
class E4Trainer(Trainer):
    def compute_loss(self, model, inputs, **kw):
        labels = inputs.pop("labels")
        weights = inputs.pop("weights")
        out = model(**inputs).logits
        sl = out[:,:-1,:]
        ll = labels[:,1:]
        ww = weights[:,1:]
        ce = torch.nn.CrossEntropyLoss(reduction="none")(sl.reshape(-1,sl.size(-1)), ll.reshape(-1)).view(ll.shape)
        mask = (ll!=-100).float()
        ww = ww*mask
        ww = ww/(ww.sum(1,keepdim=True)/mask.sum(1,keepdim=True)).clamp(min=1e-6)
        return ((ce*ww).sum()/mask.sum())

# ----------------------------
# Run
# ----------------------------
for name,cfg in STUDENTS.items():
    tok = AutoTokenizer.from_pretrained(cfg["path"], trust_remote_code=True)
    if tok.pad_token is None: tok.pad_token=tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(cfg["path"], load_in_4bit=True, device_map="auto", trust_remote_code=True)
    model = get_peft_model(model, LoraConfig(
        r=16,lora_alpha=32,target_modules=["q_proj","k_proj","v_proj","o_proj"],lora_dropout=0.05,task_type="CAUSAL_LM"
    ))
    trainer = E4Trainer(
        model=model,
        args=TrainingArguments(
            output_dir=os.path.join(OUT_DIR,name),
            num_train_epochs=EPOCHS,
            per_device_train_batch_size=BATCH,
            gradient_accumulation_steps=GRAD_ACC,
            learning_rate=LR,
            fp16=True,
            remove_unused_columns=False,
            report_to="none",
        ),
        train_dataset=E4Dataset(rows,tok,cfg["is_instruct"]),
    )
    trainer.train()
    model.save_pretrained(os.path.join(OUT_DIR,name))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Step,Training Loss


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model-00004-of-00004.safetensors:   0%|          | 0.00/3.56G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Step,Training Loss
